# Learning a swing up policy using reinforcement learning (DQN-learning)
by TimJanssen66

### General understanding
I'm using ...

## Table of contents

1. <a href="#Training-on-simulator">Training on simulator</a>
2. <a href="#Testing-on-simulator">Testing on simulator</a>
3. <a href="#Testing-on-setup">Testing on setup</a>
4. <a href="#Exercise-6:-Design-Assignment-Environment.">Exercise 6: Design Assignment Environment</a>


In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import gymnasium as gym
import gym_unbalanced_disk
from collections import deque
import random
import matplotlib.pyplot as plt
import time

class CompatibilityWrapper(gym.Wrapper):
    """
    Translates old Gym API environments (like UnbalancedDisk) 
    to the modern Gymnasium API to prevent crash errors.
    """
    def reset(self, *, seed=None, options=None):
        # 1. Intercept the reset call and strip the unexpected kwargs
        obs = self.env.reset()
        
        # 2. Ensure the output is a tuple of (observation, info_dict)
        if not isinstance(obs, tuple):
            return obs, {}
        elif len(obs) == 2 and isinstance(obs[1], dict):
            return obs
        else:
            return obs, {}

    def step(self, action):
        result = self.env.step(action)
        
        # 1. If the old environment returns 4 variables (obs, reward, done, info)
        if len(result) == 4:
            obs, reward, done, info = result
            # Map 'done' to 'terminated', and set 'truncated' to False
            return obs, reward, done, False, info
            
        # 2. If it already returns 5, just pass it through
        return result

# Hardware constraints: mapping 5 discrete actions to voltages
DISCRETE_ACTIONS = np.linspace(-3.0, 3.0, 5)
NUM_ACTIONS = len(DISCRETE_ACTIONS)


In [2]:
class DQN(nn.Module):
    def __init__(self, n_observations, n_actions):
        super(DQN, self).__init__()
        # Simple MLP: 2 inputs -> 40 -> 40 -> 5 outputs
        self.network = nn.Sequential(
            nn.Linear(n_observations, 40),
            nn.Tanh(),
            nn.Linear(40, 40),
            nn.Tanh(),
            nn.Linear(40, n_actions)
        )

    def forward(self, x):
        return self.network(x)

## Training on simulator

In [4]:
def train_dqn():
    # env = gym.make('unbalanced-disk-v0', dt=0.05)
    # env = gym.wrappers.TimeLimit(env, max_episode_steps=300)
    # 1. Instantiate the raw environment directly to bypass gym.make's strict version checkers
    base_env = gym_unbalanced_disk.UnbalancedDisk(dt=0.05)
    
    # 2. Wrap it with our new compatibility translator
    env = CompatibilityWrapper(base_env)
    
    # 3. Apply the time limit wrapper
    env = gym.wrappers.TimeLimit(env, max_episode_steps=300)
    
    # Hyperparameters
    batch_size = 64
    gamma = 0.98
    lr = 0.001
    target_update = 200 # Update target network every 200 steps
    memory_capacity = 20000
    episodes = 500
    
    # Exploration schedule
    epsilon_start = 1.0
    epsilon_end = 0.05
    epsilon_decay = 0.99
    epsilon = epsilon_start

    n_observations = env.observation_space.shape[0]
    
    policy_net = DQN(n_observations, NUM_ACTIONS)
    target_net = DQN(n_observations, NUM_ACTIONS)
    target_net.load_state_dict(policy_net.state_dict())
    
    optimizer = optim.Adam(policy_net.parameters(), lr=lr)
    memory = deque(maxlen=memory_capacity)
    
    step_count = 0
    rewards_history = []

    print("Starting DQN Training...")
    for ep in range(episodes):
        obs, _ = env.reset()
        total_reward = 0
        done = False
        
        while not done:
            # 1. Epsilon-greedy action selection
            if random.random() < epsilon:
                action_idx = random.randrange(NUM_ACTIONS)
            else:
                with torch.no_grad():
                    obs_tensor = torch.FloatTensor(obs).unsqueeze(0)
                    action_idx = policy_net(obs_tensor).argmax().item()
            
            # Map discrete index to continuous voltage
            # u_applied = np.array([DISCRETE_ACTIONS[action_idx]], dtype=np.float32)
            u_applied = float(DISCRETE_ACTIONS[action_idx])
            
            # 2. Step environment
            obs_next, reward, terminated, truncated, _ = env.step(u_applied)
            done = terminated or truncated
            total_reward += reward
            
            # 3. Store in replay memory
            memory.append((obs, action_idx, reward, obs_next, terminated))
            obs = obs_next
            
            # 4. Optimize model
            if len(memory) > batch_size:
                batch = random.sample(memory, batch_size)
                
                b_obs = torch.FloatTensor(np.array([t[0] for t in batch]))
                b_act = torch.LongTensor(np.array([t[1] for t in batch])).unsqueeze(1)
                b_rew = torch.FloatTensor(np.array([t[2] for t in batch]))
                b_nxt = torch.FloatTensor(np.array([t[3] for t in batch]))
                b_done = torch.FloatTensor(np.array([t[4] for t in batch]))
                
                # Compute current Q values
                q_values = policy_net(b_obs).gather(1, b_act).squeeze()
                
                # Compute target Q values using target_net (no gradients)
                with torch.no_grad():
                    max_next_q = target_net(b_nxt).max(1)[0]
                    target_q = b_rew + gamma * max_next_q * (1 - b_done)

                # Compute loss and update
                loss = nn.MSELoss()(q_values, target_q)
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            # 5. Update Target Network
            step_count += 1
            if step_count % target_update == 0:
                target_net.load_state_dict(policy_net.state_dict())
 
        epsilon = max(epsilon_end, epsilon_decay * epsilon)
        rewards_history.append(total_reward)

        if (ep + 1) % 50 == 0:
            avg_rew = np.mean(rewards_history[-50:])
            print(f"Episode {ep + 1}/{episodes} | Epsilon: {epsilon:.2f} | Avg Reward: {avg_rew:.2f}")

    torch.save(policy_net.state_dict(), 'dqn_weights.pth')
    print("Training complete. Weights saved to 'dqn_weights.pth'")
    env.close()

if __name__ == "__main__":
    train_dqn()

Starting DQN Training...


KeyboardInterrupt: 

## Test on simulator

In [8]:
# Define the exact same architecture

def test_dqn_simulator():
    # env = gym.make('unbalanced-disk-v0', dt=0.05, render_mode='human')
    # 1. Instantiate raw environment with the render mode flag
    base_env = gym_unbalanced_disk.UnbalancedDisk(dt=0.05, render_mode='human')
    
    # 2. Wrap it with the compatibility translator
    env = CompatibilityWrapper(base_env)
    
    n_obs = env.observation_space.shape[0]
    
    # Load network and weights
    policy_net = DQN(n_obs, NUM_ACTIONS)
    try:
        policy_net.load_state_dict(torch.load('dqn_weights.pth'))
        policy_net.eval() # Set to evaluation mode
    except FileNotFoundError:
        print("Error: 'dqn_weights.pth' not found.")
        return

    obs, _ = env.reset()
    done = False
    print("Testing DQN on Simulator...")
    
    try:
        while not done:
            # Pure greedy policy execution
            with torch.no_grad():
                obs_tensor = torch.FloatTensor(obs).unsqueeze(0)
                action_idx = policy_net(obs_tensor).argmax().item()
                
            # u_applied = np.array([DISCRETE_ACTIONS[action_idx]], dtype=np.float32)
            u_applied = float(DISCRETE_ACTIONS[action_idx])
            
            obs, reward, terminated, truncated, _ = env.step(u_applied)
            done = terminated or truncated
            
            env.render()
            time.sleep(1/24)
    finally:
        env.close()

if __name__ == "__main__":
    test_dqn_simulator()

Testing DQN on Simulator...


KeyboardInterrupt: 

## Test on setup

In [ ]:
# import hardware_unbalanced_disk # Lab specific API

def run_dqn_hardware():
    # env = hardware_unbalanced_disk.RealSetup(dt=0.05) 
    # 1. Instantiate the physical hardware connection (use the exact command your TA provides)
    base_env = hardware_unbalanced_disk.RealSetup(dt=0.05) 
    n_obs = 2 # Assuming hardware returns [angle, velocity]
    
    policy_net = DQN(n_obs, NUM_ACTIONS)
    policy_net.load_state_dict(torch.load('dqn_weights.pth'))
    policy_net.eval()
    
    obs, _ = env.reset()
    print("Starting Hardware Execution. Press Ctrl+C to emergency stop.")
    
    try:
        for step in range(300):
            # 1. State estimation (Read encoders via hardware API)
            
            # 2. Forward pass through DNN
            with torch.no_grad():
                obs_tensor = torch.FloatTensor(obs).unsqueeze(0)
                action_idx = policy_net(obs_tensor).argmax().item()
                
            # u_applied = np.array([DISCRETE_ACTIONS[action_idx]], dtype=np.float32)
            u_applied = float(DISCRETE_ACTIONS[action_idx])
            
            # 3. Apply computed voltage to the motor
            obs, reward, terminated, truncated, _ = env.step(u_applied)
            
            if terminated or truncated:
                break
                
    except KeyboardInterrupt:
        print("Emergency Stop Triggered.")
    finally:
        # Crucial: Always ground the motor to 0V when exiting
        # env.set_voltage(0.0)
        env.close()

if __name__ == "__main__":
    # run_dqn_hardware()
    pass

# Exercise 6: Design Assignment Environment

In [ ]:
!pip install git+https://github.com/MaartenSchoukens/gym-unbalanced-disk@master
# in case of error try installing pyglet 
# !pip install pyglet


In [ ]:
import gymnasium as gym
import time
import gym_unbalanced_disk #is required
import numpy as np
env = gym.make('unbalanced-disk-v0',dt=0.05) #if this fails restart the kernel or use gym_unbalanced_disk.UnbalancedDisk
env = gym_unbalanced_disk.UnbalancedDisk(dt=0.05)
# I uncommented the last env = ..., Is that the right thing to do?


In [ ]:
#if you had error just run this cell again
observation, info = env.reset()
try:
    for i in range(200):
        u = env.action_space.sample() #random input
        observation, reward, terminated, truncated, info = env.step(u) 
        print(observation, reward)
        env.render()
        time.sleep(1/24)
        if terminated or truncated:
            observation, info = env.reset()
finally: #this will always run such to close the visualization
    env.close()

    

In [ ]:
Umax = 4

# c) Input sequence
# We create an open-loop input sequence that alternates maximum control effort.
# 20 steps * 0.05s = 1.0 seconds per swing direction.
ulist = np.concatenate([
    np.full(10, Umax),   # Swing right
    np.full(10, -Umax),  # Swing left
    np.full(10, Umax),   # Swing right (building momentum)
    np.full(10, -Umax),  # Swing left  (building momentum)
]) 
# This sequence actually swings back over the top and forth over the top
obs, info = env.reset()
try:
    for u in ulist:
        obs, reward, terminated, truncated, info = env.step(u)
        print(obs,reward)
        env.render()
        time.sleep(1/24)
        if terminated or truncated:
            obs, info = env.reset()
finally: #this will always run
    env.close()
    